In [2]:
import os 
from dotenv import load_dotenv

load_dotenv()

True

In [3]:
from langchain_community.document_loaders import PyPDFLoader

pdf_path="telecom_guide.pdf"

loader=PyPDFLoader(pdf_path)
pages=loader.load()

print(f"Loaded {len(pages)} pages from the pdf");
print("\n...First page preview (first 500 chars)")
print(pages[0].page_content[:500])

C:\Users\anant\AppData\Local\Temp\ipykernel_30856\1698320833.py:1: DeprecationWarning: `langchain-community` is being sunset and is no longer actively maintained. See https://github.com/langchain-ai/langchain-community/issues/674 for details and migration guidance toward standalone integration packages.
  from langchain_community.document_loaders import PyPDFLoader
E:\LangChain\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Loaded 9 pages from the pdf

...First page preview (first 500 chars)
Telecom Technical Reference Guide  - Internal Use Only
Telecom Technical
Reference Guide
Customer Care & Network Operations Edition
Version 3.2  |  Covers 2G / 3G / 4G LTE / 5G
Page 1


In [5]:
#the pdf content is very hight..it extends the context windowsze..
#so split into fixed size chunks
#chunks are connected with some tokens common to each other

from langchain_text_splitters import RecursiveCharacterTextSplitter
splitter=RecursiveCharacterTextSplitter(
    chunk_size=600, #words (chars)---(100-150 toekns)
    chunk_overlap=100,
    separators=["\n\n","\n",".",""] 
)

chunks=splitter.split_documents(pages)

len(chunks)


37

In [7]:
#chunks[0]
chunks[0].page_content

'Telecom Technical Reference Guide  - Internal Use Only\nTelecom Technical\nReference Guide\nCustomer Care & Network Operations Edition\nVersion 3.2  |  Covers 2G / 3G / 4G LTE / 5G\nPage 1'

In [12]:
#now i did upto extract data from the source and separate into chunks...
#now my next step is to convert the chunks into meaningful vectors..

from langchain_huggingface import HuggingFaceEmbeddings #for create embeddings
from langchain_chroma import Chroma

embeddings =HuggingFaceEmbeddings(model_name="sentence-transformers/all-MiniLM-L6-v2")
vector_store=Chroma.from_documents(chunks,embeddings) #store it in the vector db

print(f"Vector store ready. {vector_store._collection.count()} vectors stored")

#here i stored the 37 chunks in the vector db in the form of vectors

Loading weights: 100%|██████████| 103/103 [00:00<00:00, 3300.73it/s]


Vector store ready. 74 vectors stored


In [14]:
#here the step i am goin to do is retriever
retriever=vector_store.as_retriever(search_kwargs={"k":3}) #k means choosing the top 3 relevant chunks which is related to the user query
test_query="What is the VolTE and how does it improve call quality?"

#retrieve chunks related to the user query
retrieved=retriever.invoke(test_query)

for i,doc in enumerate(retrieved,1):
    print(f"....chunk {i}...")
    print(doc.page_content[:300])
    print()

....chunk 1...
Telecom Technical Reference Guide  - Internal Use Only
6. VoLTE, VoWiFi, and Advanced Voice Services
Voice over LTE (VoLTE) and Voice over Wi-Fi (VoWiFi) are IP-based voice technologies that replace the legacy
circuit-switched voice channel used in 2G and 3G networks.
VoLTE: With VoLTE, voice calls 

....chunk 2...
Telecom Technical Reference Guide  - Internal Use Only
6. VoLTE, VoWiFi, and Advanced Voice Services
Voice over LTE (VoLTE) and Voice over Wi-Fi (VoWiFi) are IP-based voice technologies that replace the legacy
circuit-switched voice channel used in 2G and 3G networks.
VoLTE: With VoLTE, voice calls 

....chunk 3...
voice simultaneously without degradation. VoLTE requires a compatible device, a VoLTE-enabled SIM, and an
account that has VoLTE activated.
Enabling VoLTE: On most Android devices navigate to Settings > Mobile Network > VoLTE and toggle it on. On
iPhone go to Settings > Mobile Data > Mobile Data Opt



In [ ]:
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.output_parsers import StrOutputParser
from langchain_core.runnables import RunnablePassthrough
from langchain_groq import ChatGroq
from langchain_google_genai import ChatGoogleGenerativeAI

#a helper function to join retrieved chunks into a single context string
def format_docs(docs):
    return "\n\n...\n\n".join(doc.page_content for doc in docs)

    
#its like giving instructions to the llm
SYSTEM_PROMPT = """\
You are a helpful telecom assistant.
Answerthe question using only the context provided below.
If the context does not contains enough information ,say not clear content,

context:
{context}
"""

prompt=ChatPromptTemplate.from_messages([
    ("system",SYSTEM_PROMPT),
    ("human","{question}")
])

#llm via 
llm = ChatGoogleGenerativeAI(model="gemma-4-31b-it",temperature=0)

chain=(
    {"context":retriever | format_docs,"question":RunnablePassthrough()}
    | prompt
    | llm
    | StrOutputParser()
)

print("RAG chain assembled")

RAG chain assembled


In [26]:
question="How does international roaming work and what charges should I expect"

print(f"Q :{question}\n")
print("A : ",chain.invoke(question))

Q :How does international roaming work and what charges should I expect

A :  International roaming works when a customer travels outside their home network's coverage area and their device connects to a partner network in the visited country. Technically, the visited network authenticates the customer using an inter-operator signalling protocol (SS7 or Diameter), and the home network authorises the service and validates the subscription. All SMS, voice, and data traffic is then tunnelled back to the home network for billing.

Regarding charges, the network uses roaming zones:
* **Zone A (EU, UK, Australia, New Zealand):** Lowest roaming rates.
* **Zone B (USA, Canada, Japan, Singapore):** Moderate rates.
* **Zone C (Rest of World):** Highest per-minute and per-MB charges.
